In [19]:
from pylibCZIrw import czi as pyczi
import matplotlib.pyplot as plt 
import matplotlib.cm as cm
import os
from skimage.morphology import skeletonize
from scipy import ndimage
from scipy.spatial import cKDTree
from skimage.morphology import skeletonize
from skimage.measure import label

In [24]:
def find_skeleton_endpoints(skeleton: np.ndarray) -> np.ndarray:
    """
    Return (N, 2) array of [row, col] positions that are skeleton
    pixels with exactly ONE 8-connected neighbour.
    """
    kernel = np.ones((3, 3), dtype=np.uint8)
    kernel[1, 1] = 0

    neighbour_count = ndimage.convolve(
        skeleton.astype(np.uint8), kernel, mode='constant', cval=0
    )
    mask = (skeleton > 0) & (neighbour_count == 1)
    return np.argwhere(mask)


# ──────────────────────────────────────────────────────────────
#  2.  BROKEN-LINE DETECTION
# ──────────────────────────────────────────────────────────────

def find_broken_lines(skeleton: np.ndarray,
                      min_gap: int = 10,
                      max_gap: int = 50) -> tuple[list[dict], np.ndarray]:
    """
    Detect broken lines in a skeletonised binary image.

    A 'break' is flagged when two endpoints:
      •  belong to DIFFERENT connected components
      •  gap distance is within [min_gap, max_gap]

    Parameters
    ----------
    skeleton : 2-D binary array
    min_gap  : minimum pixel distance — filters out tiny skeletonisation
               artefacts and near-touching segments (higher res → raise this)
    max_gap  : maximum pixel distance — upper bound on what counts as a break

    Returns
    -------
    breaks    : list of dicts  {'pt1', 'pt2', 'distance'}
    endpoints : (N, 2) array of ALL endpoint coordinates
    """
    skeleton  = skeleton.astype(bool)
    endpoints = find_skeleton_endpoints(skeleton)

    if len(endpoints) < 2:
        return [], endpoints

    comp_map  = label(skeleton, connectivity=2)
    ep_labels = np.array([comp_map[r, c] for r, c in endpoints])

    # Query only pairs within max_gap — min_gap filtered afterwards
    tree            = cKDTree(endpoints)
    candidate_pairs = tree.query_pairs(r=max_gap)

    breaks = []
    for i, j in candidate_pairs:
        dist = float(np.linalg.norm(endpoints[i] - endpoints[j]))

        if dist < min_gap:                          # ← skip tiny artefacts
            continue
        if ep_labels[i] == ep_labels[j]:            # ← same component, skip
            continue

        breaks.append(dict(
            pt1      = tuple(endpoints[i]),
            pt2      = tuple(endpoints[j]),
            distance = dist
        ))

    breaks.sort(key=lambda b: b["distance"])
    return breaks, endpoints


# ──────────────────────────────────────────────────────────────
#  3.  VISUALISATION
# ──────────────────────────────────────────────────────────────

def print_broken_lines(directory: str,
                   min_gap: int = 10,
                   max_gap: int = 50) -> None:
    """
    Load every .czi file in *directory*, skeletonise it, detect broken
    lines in the [min_gap, max_gap] range, and display annotated figures.

    Parameters
    ----------
    directory : path to folder containing .czi files
    min_gap   : ignore gaps SMALLER than this (skeletonisation noise)
    max_gap   : ignore gaps LARGER  than this (too far apart to be a break)
    """
    files = sorted(f for f in os.listdir(directory) if f.endswith(".czi"))
    n = len(files)

    if n == 0:
        print("No .czi files found in the directory.")
        return

    cols = min(3, n)
    rows = (n + cols - 1) // cols


    for i, fname in enumerate(files):
        path = os.path.join(directory, fname)

        # ── load ────────────────────────────────────────────────────
        with pyczi.open_czi(path) as czidoc:
            raw = czidoc.read()

        img = np.squeeze(raw)
        if img.ndim > 2:
            img = img[0]

        # ── skeletonise ─────────────────────────────────────────────
        skeleton = skeletonize(img > 0)

        # ── detect breaks ───────────────────────────────────────────
        breaks, endpoints = find_broken_lines(skeleton,
                                              min_gap=min_gap,
                                              max_gap=max_gap)
        n_breaks   = len(breaks)
        n_endpoint = len(endpoints)

        # ── build RGB canvas ────────────────────────────────────────
        canvas = np.zeros((*skeleton.shape, 3), dtype=np.uint8)
        canvas[skeleton] = (200, 200, 200)


        # ── console summary ─────────────────────────────────────────
        print(f"{'─' * 55}")
        print(f"File              : {fname}")
        print(f"Total endpoints   : {n_endpoint}")
        print(f"Broken lines      : {n_breaks}  (gap range [{min_gap}–{max_gap}] px)")
        if breaks:
            ds = [b["distance"] for b in breaks]
            print(f"Gap sizes (px)    : "
                  f"min={min(ds):.1f}  mean={np.mean(ds):.1f}  max={max(ds):.1f}")



def show_broken_lines(directory: str,
                   min_gap: int = 10,
                   max_gap: int = 50) -> None:
    """
    Load every .czi file in *directory*, skeletonise it, detect broken
    lines in the [min_gap, max_gap] range, and display annotated figures.

    Parameters
    ----------
    directory : path to folder containing .czi files
    min_gap   : ignore gaps SMALLER than this (skeletonisation noise)
    max_gap   : ignore gaps LARGER  than this (too far apart to be a break)
    """
    files = sorted(f for f in os.listdir(directory) if f.endswith(".czi"))
    n = len(files)

    if n == 0:
        print("No .czi files found in the directory.")
        return

    cols = min(3, n)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 6 * rows))
    axes = np.array(axes, dtype=object).ravel()

    for i, fname in enumerate(files):
        path = os.path.join(directory, fname)

        # ── load ────────────────────────────────────────────────────
        with pyczi.open_czi(path) as czidoc:
            raw = czidoc.read()

        img = np.squeeze(raw)
        if img.ndim > 2:
            img = img[0]

        # ── skeletonise ─────────────────────────────────────────────
        skeleton = skeletonize(img > 0)

        # ── detect breaks ───────────────────────────────────────────
        breaks, endpoints = find_broken_lines(skeleton,
                                              min_gap=min_gap,
                                              max_gap=max_gap)
        n_breaks   = len(breaks)
        n_endpoint = len(endpoints)

        # ── build RGB canvas ────────────────────────────────────────
        canvas = np.zeros((*skeleton.shape, 3), dtype=np.uint8)
        canvas[skeleton] = (200, 200, 200)

        ax = axes[i]
        ax.imshow(canvas)
        ax.set_title(
            f"{fname}\n"
            f"Broken lines: {n_breaks}  |  Endpoints: {n_endpoint}  |  "
            f"Gap range: [{min_gap}, {max_gap}] px",
            fontsize=9
        )
        ax.axis("off")

        # ── all endpoints – small cyan dots ─────────────────────────
        if n_endpoint:
            ax.scatter(endpoints[:, 1], endpoints[:, 0],
                       s=14, c="cyan", zorder=3, alpha=0.7)

        # ── each break: red hollow circles + yellow dashed gap ──────
        for b in breaks:
            (r1, c1), (r2, c2) = b["pt1"], b["pt2"]

            for rc in [(r1, c1), (r2, c2)]:
                ax.plot(rc[1], rc[0],
                        "o", color="red", markersize=11,
                        markerfacecolor="none", markeredgewidth=2, zorder=5)

            ax.plot([c1, c2], [r1, r2],
                    "--", color="yellow", linewidth=2, alpha=0.9, zorder=4)

            ax.text((c1 + c2) / 2, (r1 + r2) / 2,
                    f" {b['distance']:.1f}px",
                    color="yellow", fontsize=6, zorder=6,
                    va="center", ha="left")

        # ── legend ──────────────────────────────────────────────────
        legend = [
            Line2D([0], [0], marker="o", color="w",
                   markerfacecolor="cyan", markersize=7,
                   label=f"All endpoints  ({n_endpoint})"),
            Line2D([0], [0], marker="o", color="red", markersize=9,
                   markerfacecolor="none", markeredgewidth=2,
                   label="Break endpoints"),
            Line2D([0], [0], color="yellow", linestyle="--",
                   label=f"Gaps  ({n_breaks})"),
        ]
        ax.legend(handles=legend, loc="upper right",
                  fontsize=7, framealpha=0.6)

        # ── console summary ─────────────────────────────────────────
        print(f"{'─' * 55}")
        print(f"File              : {fname}")
        print(f"Total endpoints   : {n_endpoint}")
        print(f"Broken lines      : {n_breaks}  (gap range [{min_gap}–{max_gap}] px)")
        if breaks:
            ds = [b["distance"] for b in breaks]
            print(f"Gap sizes (px)    : "
                  f"min={min(ds):.1f}  mean={np.mean(ds):.1f}  max={max(ds):.1f}")

    for j in range(n, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(
        f"Broken-line detection  —  gap window: [{min_gap}, {max_gap}] px",
        fontsize=13, y=1.01
    )
    plt.tight_layout()
    plt.show()

In [25]:
directory_1 = "/Users/ctuna/Desktop/axon_codes/data/x63 bmi tubulin iii_data/"

directory_2 = "/Users/ctuna/Desktop/axon_codes/data/x63 Ni tubulin_data/"

directory_3 = "/Users/ctuna/Desktop/axon_codes/data/x63 scr tubIII ID8221225_data/"

In [29]:
print_broken_lines(directory_1, min_gap=40, max_gap=100)

───────────────────────────────────────────────────────
File              : New-01.czi
Total endpoints   : 9804
Broken lines      : 428330  (gap range [40–100] px)
Gap sizes (px)    : min=40.0  mean=73.7  max=100.0
───────────────────────────────────────────────────────
File              : New-02.czi
Total endpoints   : 6644
Broken lines      : 239777  (gap range [40–100] px)
Gap sizes (px)    : min=40.0  mean=73.5  max=100.0
───────────────────────────────────────────────────────
File              : New-03.czi
Total endpoints   : 5252
Broken lines      : 158472  (gap range [40–100] px)
Gap sizes (px)    : min=40.0  mean=72.7  max=100.0
───────────────────────────────────────────────────────
File              : New-04.czi
Total endpoints   : 2970
Broken lines      : 53549  (gap range [40–100] px)
Gap sizes (px)    : min=40.0  mean=72.5  max=100.0
───────────────────────────────────────────────────────
File              : New-05.czi
Total endpoints   : 7105
Broken lines      : 228983  (

In [30]:
print_broken_lines(directory_2, min_gap=40, max_gap=100)

───────────────────────────────────────────────────────
File              : New-01.czi
Total endpoints   : 9471
Broken lines      : 500019  (gap range [40–100] px)
Gap sizes (px)    : min=40.0  mean=73.1  max=100.0
───────────────────────────────────────────────────────
File              : New-02.czi
Total endpoints   : 10430
Broken lines      : 478331  (gap range [40–100] px)
Gap sizes (px)    : min=40.0  mean=73.5  max=100.0
───────────────────────────────────────────────────────
File              : New-03.czi
Total endpoints   : 5077
Broken lines      : 215248  (gap range [40–100] px)
Gap sizes (px)    : min=40.0  mean=73.2  max=100.0
───────────────────────────────────────────────────────
File              : New-04.czi
Total endpoints   : 12764
Broken lines      : 671598  (gap range [40–100] px)
Gap sizes (px)    : min=40.0  mean=74.1  max=100.0


In [31]:
print_broken_lines(directory_3, min_gap=40, max_gap=100)

───────────────────────────────────────────────────────
File              : New-01.czi
Total endpoints   : 8411
Broken lines      : 232452  (gap range [40–100] px)
Gap sizes (px)    : min=40.0  mean=74.0  max=100.0
───────────────────────────────────────────────────────
File              : New-02.czi
Total endpoints   : 19282
Broken lines      : 1542135  (gap range [40–100] px)
Gap sizes (px)    : min=40.0  mean=74.1  max=100.0
───────────────────────────────────────────────────────
File              : New-03.czi
Total endpoints   : 14201
Broken lines      : 1066045  (gap range [40–100] px)
Gap sizes (px)    : min=40.0  mean=73.7  max=100.0
───────────────────────────────────────────────────────
File              : New-04.czi
Total endpoints   : 13740
Broken lines      : 1014226  (gap range [40–100] px)
Gap sizes (px)    : min=40.0  mean=73.4  max=100.0
